# Entrenamiento en Kaggle — Mario 1-1 (PPO principal, DQN opcional)

**Antes de empezar (panel derecho → Session options):**
- *Accelerator*: GPU (P100 o T4). El emulador corre en CPU; la GPU acelera las actualizaciones de la red.
- *Internet*: **On** (hace falta para `git clone` y `pip`; Kaggle puede pedir verificar el teléfono).
- Para entrenar horas sin tener la pestaña abierta: **Save Version → Save & Run All (Commit)**. Las sesiones duran como máximo ~12 h y la cuota de GPU es ~30 h/semana; una sesión interactiva inactiva también la consume.
- Todo lo que quieras conservar debe quedar en `/kaggle/working` (por eso las salidas van a `/kaggle/working/out`).

*Notebook sin probar en Kaggle: si una celda falla, copia el error completo.*

In [ ]:
# 1) Hardware: ANOTA los núcleos (nproc) y la RAM; n_envs de PPO debe ser <= núcleos
!nproc
!free -g
!nvidia-smi
import os, torch
print("CPU:", os.cpu_count(), "| CUDA:", torch.cuda.is_available())

In [ ]:
# 2) Código del proyecto
REPO_URL = "https://github.com/David-AGL/mario-bros-reinforcement-learning.git"
BRANCH = "david/kaggle-training"
import os
if not os.path.exists("/kaggle/working/mario/.git"):
    !git clone -b {BRANCH} {REPO_URL} /kaggle/working/mario
else:
    !git -C /kaggle/working/mario pull origin {BRANCH}
%cd /kaggle/working/mario
!git log --oneline -1

In [ ]:
# 3) Dependencias (mismas versiones fijadas del proyecto; Kaggle trae TF/JAX que rompen con numpy<2)
!pip uninstall -y -q tensorflow tensorflow-cpu jax jaxlib opencv-python opencv-contrib-python 2>/dev/null
!pip install -q "numpy<2" "gym==0.26.2" "nes-py==8.2.1" "gym-super-mario-bros==7.4.0" "gymnasium==0.29.1" "stable-baselines3==2.3.2" "opencv-python-headless<=4.10.0.84" psutil tensorboard
!python -c "from stable_baselines3 import DQN, PPO; print('SB3 OK')"
# Si pip avisa de conflictos de NumPy con otros paquetes: ignorar. Si pide reiniciar, reinicia y vuelve a ejecutar desde la celda 2.

In [ ]:
# 4) Smoke test del entorno (debe terminar en SMOKE TEST OK; anota los FPS)
!python -W ignore scripts/smoke_test_env.py

## Configuración
Receta "v3": env-v2 + recompensa escalada x0.01 (solo para el algoritmo). Pasos y checkpoints:
fija `TOTAL_STEPS` según los pasos por segundo medidos en la prueba corta (celda 6).

In [ ]:
import os
ALGO = "ppo"                       # "ppo" (recomendado) o "dqn"
SEED = 1
TOTAL_STEPS = 50_000               # PRUEBA CORTA primero; luego 5_000_000 (ver celda 6)
RUN_NAME = f"{ALGO}_s{SEED}_test"  # cámbialo en la corrida real: p. ej. f"{ALGO}_s{SEED}"
N_ENVS = os.cpu_count()            # PPO: tantos como núcleos
REWARD_SCALE = 0.01

OUT = "/kaggle/working/out"
MODELS_ROOT = f"{OUT}/models"
LOGS_ROOT = f"{OUT}/runs"

In [ ]:
# 5) Dry-run: muestra la configuración sin entrenar
!python -W ignore -m ppo.train --seed {SEED} --total-steps {TOTAL_STEPS} --n-envs {N_ENVS} --run-name {RUN_NAME} \
    --models-root {MODELS_ROOT} --logs-root {LOGS_ROOT} --reward-scale {REWARD_SCALE} --dry-run

## Prueba corta (obligatoria antes de la corrida larga)
Con `TOTAL_STEPS = 50_000` y `RUN_NAME = ppo_s1_test`, ejecuta la celda 6, mira los **pasos por segundo (fps)** del log y calcula:
`horas = TOTAL_REAL / fps / 3600`. Si el tiempo cabe en una sesión de ~12 h, lanza la corrida real (cambia `TOTAL_STEPS` y `RUN_NAME`).
Si no cabe, usa checkpoints y reanuda en otra sesión (celda 7).

In [ ]:
# 6) ENTRENAR PPO
!python -W ignore -m ppo.train --seed {SEED} --total-steps {TOTAL_STEPS} --n-envs {N_ENVS} --run-name {RUN_NAME} \
    --models-root {MODELS_ROOT} --logs-root {LOGS_ROOT} --reward-scale {REWARD_SCALE} --device auto

In [ ]:
# 6b) ALTERNATIVA: DQN con los ajustes propuestos (ejecutar en lugar de la 6, no a la vez con ella en la misma CPU)
# !python -W ignore -m dqn.train --seed {SEED} --total-steps 3000000 --run-name dqn_s{SEED}_v3 \
#     --models-root {MODELS_ROOT} --logs-root {LOGS_ROOT} --reward-scale {REWARD_SCALE} \
#     --buffer-size 200000 --optimize-memory-usage --target-update-interval 2000 \
#     --exploration-fraction 0.15 --exploration-final-eps 0.02 --checkpoint-freq 50000

In [ ]:
# 7) REANUDAR en otra sesión: sube /kaggle/working/out/models/... como Kaggle Dataset, añádelo al notebook
#    (Add Input) y apunta CKPT al último checkpoint. Usa el MISMO RUN_NAME y TOTAL_STEPS.
CKPT = "/kaggle/input/NOMBRE_DEL_DATASET/ppo_s1_2500000_steps.zip"   # <- EDITAR
# !python -W ignore -m ppo.train --seed {SEED} --total-steps {TOTAL_STEPS} --n-envs {N_ENVS} --run-name {RUN_NAME} \
#     --models-root {MODELS_ROOT} --logs-root {LOGS_ROOT} --reward-scale {REWARD_SCALE} --resume {CKPT}

## Validación de checkpoints
Evalúa solo algunos checkpoints (cada uno son 20 episodios; puede tardar varios minutos por checkpoint).
Semillas **100–119** para elegir; las **200–209** se reservan para la evaluación final.
También puedes descargar los checkpoints y evaluarlos en tu equipo (solo inferencia, CPU).

In [ ]:
import os
CKPT_DIR = f"{MODELS_ROOT}/ppo/{RUN_NAME}"
CHOSEN_STEPS = [1_000_000, 2_000_000, 3_000_000, 4_000_000, 5_000_000]   # <- EDITAR según lo que exista
for steps in CHOSEN_STEPS:
    cp = f"{CKPT_DIR}/ppo_s{SEED}_{steps}_steps.zip"
    if not os.path.exists(cp):
        print("No existe:", cp); continue
    !python -W ignore evaluate.py --algo ppo --model "{cp}" --seeds 100-119 --out results/val --tag {RUN_NAME}_{steps//1000}k

In [ ]:
# 8) Empaquetar todo lo que hay que descargar (pestaña Output) y mostrar el tamaño
!cd /kaggle/working && zip -qr mario_out.zip out mario/results && ls -lh mario_out.zip

## Después de descargar
En tu repo: `models/ppo/<run>/...` (NO va a Git: los `.zip` están ignorados), `runs/ppo/<run>/episodes.csv`,
`models/ppo/<run>/config.json` y `results/val/*.csv` (estos tres sí se versionan).